### What to do

- Import data of a tracker and his top 20 components options
- Calculate the vol of the tracker, and the implied vol of the components
- Watch the spread
- Calculate options greeks
- Compute Short straddle and long straddle pay-offs and greeks
- Watch PNL

### For Delta Hedging
- Download Underlying's data
- Dynamic hedge daily


In [2]:
import requests
import pandas as pd 
import numpy as np
import yfinance as yf

In [3]:
# Tracker data
tracker = yf.download(tickers = '^SPX', start = '2026-01-01', interval = '1d')['Close']
tracker.tail(5)

C:\Users\Romain\AppData\Local\Temp\ipykernel_1340\3146965524.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  tracker = yf.download(tickers = '^SPX', start = '2026-01-01', interval = '1d')['Close']
[*********************100%***********************]  1 of 1 completed


Ticker,^SPX
Date,
2026-04-27,7173.910156
2026-04-28,7138.799805
2026-04-29,7135.950195
2026-04-30,7209.009766
2026-05-01,7230.120117


In [4]:
# Top 20 components + Weight
compo_weight = pd.read_excel(r'C:\Users\Romain\Desktop\Codes\datasets\S&P components weight.xlsx')
compo_weight
print(compo_weight)


                          Name    Ticker    Weight
0                  NVIDIA CORP      NVDA  8.044769
1                    APPLE INC      AAPL  6.556516
2               MICROSOFT CORP      MSFT  5.255533
3               AMAZON.COM INC      AMZN  4.078155
4                 BROADCOM INC      AVGO  3.275963
..                         ...       ...       ...
499  BROWN FORMAN CORP CLASS B      BF.B  0.008046
500     THE CAMPBELL S COMPANY       CPB  0.006758
501        NEWS CORP   CLASS B       NWS  0.006111
502    PARAMOUNT SKYDANCE CL B      PSKY  0.006041
503     CONTRA HOLOGIC INCORPO  2602335D  0.000004

[504 rows x 3 columns]


In [55]:
# Implied vol for 1 month maturity (components)

def get_atm_iv(symbol, dte_target=30):
    try:
        tk = yf.Ticker(symbol)
        spot = tk.fast_info["last_price"]
        exps = tk.options
        
        today = pd.Timestamp.today()
        dtes = [(pd.Timestamp(e) - today).days for e in exps]
        best_exp = exps[np.argmin([abs(d - dte_target) for d in dtes])]
        
        chain = tk.option_chain(best_exp)
        calls, puts = chain.calls, chain.puts
        
        atm_c = calls.iloc[(calls["strike"] - spot).abs().argsort()[:1]]
        atm_p = puts.iloc[(puts["strike"] - spot).abs().argsort()[:1]]
        
        iv = (atm_c["impliedVolatility"].values[0] + 
              atm_p["impliedVolatility"].values[0]) / 2
        
        return {"symbol": symbol, "spot": spot, "expiry": best_exp, "iv_atm": iv}
    
    except Exception as e:
        return {"symbol": symbol, "error": str(e)}

results = [get_atm_iv(sym) for sym in compo_weight['Ticker']]
iv_df = pd.DataFrame(results)
print(iv_df)

$-: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")
$-: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
$BF.B: possibly delisted; no price data found  (period=1y)
$BF.B: possibly delisted; no price data found  (period=5d)
$2602335D: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")
$2602335D: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")


       symbol        spot      expiry    iv_atm  \
0        NVDA  198.449997  2026-06-05  0.425207   
1        AAPL  280.140015  2026-06-05  0.231972   
2        MSFT  414.440002  2026-06-05  0.273323   
3        AMZN  268.260010  2026-06-05  0.292915   
4        AVGO  421.279999  2026-06-05  0.519078   
..        ...         ...         ...       ...   
499      BF.B         NaN         NaN       NaN   
500       CPB   20.730000  2026-06-05  0.426275   
501       NWS   30.410000  2026-05-15  0.605473   
502      PSKY   11.090000  2026-06-05  0.538579   
503  2602335D         NaN         NaN       NaN   

                                          error  
0                                           NaN  
1                                           NaN  
2                                           NaN  
3                                           NaN  
4                                           NaN  
..                                          ...  
499  attempt to get argmin of an empt

In [56]:
# Implied Vol for tracker

tracker_iv = [get_atm_iv('^SPX')]
tracker_iv_df = pd.DataFrame(tracker_iv)
print(tracker_iv_df)



  symbol         spot      expiry    iv_atm
0   ^SPX  7230.120117  2026-06-02  0.137564


In [57]:
iv_df2 = iv_df.copy()
iv_df2 = iv_df2[['symbol', 'spot', 'expiry', 'iv_atm']]
iv_df2 = iv_df2.dropna()
iv_df2.rename(columns={'symbol': 'Ticker'}, inplace=True)

# Implied correlation
merged = compo_weight.merge(iv_df2, on='Ticker', how='inner')

weights = merged["Weight"].values / 100
sigmas  = merged["iv_atm"].values
sigma_I = float(tracker_iv_df['iv_atm'].iloc[0])

num = sigma_I**2 - np.sum(weights**2 * sigmas**2)

# Dénominateur : somme des termes croisés
denom = 0
n = len(weights)
for i in range(n):
    for j in range(n):
        if i != j:
            denom += weights[i] * weights[j] * sigmas[i] * sigmas[j]

rho_imp = num / denom
print(f"Implied Correlation : {rho_imp:.4f}")

Implied Correlation : 0.1345


In [58]:
# Realized correlation

symbols = iv_df2["Ticker"].tolist()

prices = yf.download(symbols, period="21d", auto_adjust=True)["Close"]
returns = prices.pct_change().dropna()

corr_matrix = returns.corr().values

mask = ~np.eye(n, dtype=bool)
rho_real = corr_matrix[mask].mean()
print(f"Realized Correlation: {rho_real:.4f}")



[*********************100%***********************]  468 of 468 completed


Realized Correlation: 0.1486


In [59]:
# Signal

spread = rho_imp - rho_real
print(f"Spread : {spread:.4f}")

if spread > 0.10:  # To calibrate
    print("Signal : SHORT straddle on tracker / LONG straddle on components")
elif spread < -0.05:
    print("Signal : no trade")
else:
    print("Signal : neutral, wait")

Spread : -0.0141
Signal : neutral, wait
